# Solving RL Problems: Defining States, Actions, and Rewards

## Learning Objectives

By the end of this notebook, you will be able to:
- define a state space for a simple RL problem
- define valid actions for an agent
- design a reward function that encourages the right behavior
- explain why bad reward design can create bad learning
- run a simple interaction loop before using a full RL algorithm

## Prerequisites

Before starting:
- complete `04_openai_gym_setup.ipynb`
- understand the basic RL loop: state -> action -> reward -> next state
- be comfortable reading simple Python functions

## Why this notebook matters

Before choosing any algorithm, you must first decide:

1. What should the agent observe?
2. What can the agent do?
3. What should count as success or failure?

If these are designed poorly, even a strong RL algorithm can learn the wrong thing.

## Lesson Brief

This lesson teaches students how to **formulate** a reinforcement learning problem correctly.

The focus is not on fancy algorithms, but on making good choices for states, actions, and rewards so that learning becomes possible.

Why this matters: many RL failures come from bad problem formulation, not from bad algorithms.

This notebook helps students think like environment designers, not just code writers.

## Quick Check Before You Begin

Make sure you can answer these questions:

1. Is a **state** the same thing as the real world itself?
2. Can a reward function accidentally encourage the wrong behavior?
3. Why do we need an interaction loop before training an agent?

If not, slow down and answer them before moving on.


In [1]:
%pip install "gymnasium[classic_control]" numpy matplotlib -q
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

print("✅ Libraries imported!")
print("\nSolving RL Problems: States, Actions, and Rewards")
print("=" * 60)

You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.


Note: you may need to restart the kernel to use updated packages.


✅ Libraries imported!

Solving RL Problems: States, Actions, and Rewards


## Part 1: Defining State Spaces


In [2]:
print("=" * 60)
print("Part 1: Defining State Spaces")
print("=" * 60)

examples = {
    "Grid navigation": {
        "good_state": "(row, col) position of the agent",
        "too_small": "only row",
        "too_large": "every detail of the whole computer screen",
    },
    "CartPole": {
        "good_state": "cart position, cart velocity, pole angle, pole angular velocity",
        "too_small": "pole angle only",
        "too_large": "full rendered image when not needed for this level",
    },
    "Inventory control": {
        "good_state": "current stock level and maybe demand signal",
        "too_small": "stock level from yesterday only",
        "too_large": "every warehouse detail regardless of relevance",
    },
}

for problem, details in examples.items():
    print(f"\n{problem}")
    print(" Good state representation:", details["good_state"])
    print(" Too small:", details["too_small"])
    print(" Too large:", details["too_large"])

print("\nCheckpoint: a useful state contains enough information for decision-making,")
print("but not so much unnecessary detail that the problem becomes harder to learn.")

Part 1: Defining State Spaces

Grid navigation
 Good state representation: (row, col) position of the agent
 Too small: only row
 Too large: every detail of the whole computer screen

CartPole
 Good state representation: cart position, cart velocity, pole angle, pole angular velocity
 Too small: pole angle only
 Too large: full rendered image when not needed for this level

Inventory control
 Good state representation: current stock level and maybe demand signal
 Too small: stock level from yesterday only
 Too large: every warehouse detail regardless of relevance

Checkpoint: a useful state contains enough information for decision-making,
but not so much unnecessary detail that the problem becomes harder to learn.


## Part 2: Defining Action Spaces


In [3]:
print("\n" + "=" * 60)
print("Part 2: Defining Action Spaces")
print("=" * 60)

# CartPole actions
env = gym.make("CartPole-v1")
print("\nCartPole-v1")
print(f" Action space: {env.action_space}")
print(" Valid actions:")
print(" 0 -> push cart left")
print(" 1 -> push cart right")

# FrozenLake actions
env_fl = gym.make("FrozenLake-v1")
obs_fl, info = env_fl.reset()
print("\nFrozenLake-v1")
print(f" Action space: {env_fl.action_space}")
print(" Valid actions:")
print(" 0 -> left")
print(" 1 -> down")
print(" 2 -> right")
print(" 3 -> up")

env.close()
env_fl.close()

print("\nAction-space design rules:")
print("- The actions must be valid choices the agent can actually take.")
print("- Too few actions makes the agent powerless.")
print("- Too many unnecessary actions makes learning harder.")


Part 2: Defining Action Spaces

CartPole-v1
 Action space: Discrete(2)
 Valid actions:
 0 -> push cart left
 1 -> push cart right

FrozenLake-v1
 Action space: Discrete(4)
 Valid actions:
 0 -> left
 1 -> down
 2 -> right
 3 -> up

Action-space design rules:
- The actions must be valid choices the agent can actually take.
- Too few actions makes the agent powerless.
- Too many unnecessary actions makes learning harder.


## Part 3: Designing Reward Functions


In [4]:
print("\n" + "=" * 60)
print("Part 3: Designing Reward Functions")
print("=" * 60)

reward_examples = [
    ("Maze goal", "+10 for goal, -1 per move, -10 for falling into a trap", "Encourages fast and safe navigation"),
    ("CartPole", "+1 per balanced step", "Encourages keeping the pole upright longer"),
    ("Bad driving reward", "+100 for speed only", "May encourage unsafe driving and crashes"),
]

for task, reward_design, effect in reward_examples:
    print(f"\nTask: {task}")
    print(" Reward design:", reward_design)
    print(" Likely effect:", effect)

print("\nReward-design checklist:")
print("- Does the reward reflect the real goal?")
print("- Can the agent exploit the reward without solving the task?")
print("- Does the reward encourage safe and sensible behavior?")


Part 3: Designing Reward Functions

Task: Maze goal
 Reward design: +10 for goal, -1 per move, -10 for falling into a trap
 Likely effect: Encourages fast and safe navigation

Task: CartPole
 Reward design: +1 per balanced step
 Likely effect: Encourages keeping the pole upright longer

Task: Bad driving reward
 Reward design: +100 for speed only
 Likely effect: May encourage unsafe driving and crashes

Reward-design checklist:
- Does the reward reflect the real goal?
- Can the agent exploit the reward without solving the task?
- Does the reward encourage safe and sensible behavior?


## Part 4: Running RL Simulations


In [5]:
print("\n" + "=" * 60)
print("Part 4: Running RL Simulations")
print("=" * 60)

env = gym.make("FrozenLake-v1", is_slippery=False)
obs, info = env.reset(seed=7)
total_reward = 0

print("Starting state:", obs)
for step_idx in range(8):
    action = env.action_space.sample()
    next_obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    print(
        f"Step {step_idx + 1}: state={obs}, action={action}, next_state={next_obs}, "
        f"reward={reward}, terminated={terminated}, truncated={truncated}"
    )
    obs = next_obs
    if terminated or truncated:
        print("Episode ended. Reset would be needed before a new run.")
        break

env.close()
print("\nTotal reward collected in this random simulation:", total_reward)
print("This is the full interaction loop that later algorithms will use repeatedly.")


Part 4: Running RL Simulations
Starting state: 0
Step 1: state=0, action=0, next_state=0, reward=0.0, terminated=False, truncated=False
Step 2: state=0, action=0, next_state=0, reward=0.0, terminated=False, truncated=False
Step 3: state=0, action=1, next_state=4, reward=0.0, terminated=False, truncated=False
Step 4: state=4, action=2, next_state=5, reward=0.0, terminated=True, truncated=False
Episode ended. Reset would be needed before a new run.

Total reward collected in this random simulation: 0.0
This is the full interaction loop that later algorithms will use repeatedly.


## Worked Example: From Business Problem to RL Formulation

Suppose you want to manage inventory in a small store.

A useful RL formulation could be:

- **State:** current stock level
- **Action:** reorder or do not reorder
- **Reward:** sales profit minus shortage penalty minus restocking cost

This is a good teaching example because students can clearly see how state,
action, and reward connect to a real decision problem.

In [6]:
import numpy as np

np.random.seed(42)

MAX_STOCK = 10
ACTIONS = {0: "do not reorder", 1: "reorder 5 units"}


def inventory_step(state, action):
    stock = min(state + (5 if action == 1 else 0), MAX_STOCK)
    demand = np.random.randint(0, 4)
    sold = min(stock, demand)
    shortage_penalty = 2 if demand > stock else 0
    reorder_cost = 0.5 if action == 1 else 0.0
    reward = sold - shortage_penalty - reorder_cost
    next_state = max(stock - demand, 0)
    return next_state, reward, demand


state = 3
print("Starting stock level:", state)
for day in range(5):
    action = np.random.randint(0, 2)
    next_state, reward, demand = inventory_step(state, action)
    print(
        f"Day {day + 1}: action='{ACTIONS[action]}', demand={demand}, "
        f"next_state={next_state}, reward={reward:.1f}"
    )
    state = next_state

print("\nTeaching point:")
print("This example is not about advanced RL yet.")
print("It is about translating a real problem into states, actions, and rewards.")

Starting stock level: 3
Day 1: action='do not reorder', demand=3, next_state=0, reward=3.0
Day 2: action='do not reorder', demand=2, next_state=0, reward=-2.0
Day 3: action='do not reorder', demand=3, next_state=0, reward=-2.0
Day 4: action='do not reorder', demand=0, next_state=0, reward=0.0
Day 5: action='do not reorder', demand=1, next_state=0, reward=-2.0

Teaching point:
This example is not about advanced RL yet.
It is about translating a real problem into states, actions, and rewards.


## Summary

The main lesson of this notebook is that RL problem design comes before RL algorithms.

### Remember

- **State** should contain useful decision information.
- **Action** should reflect the real choices available to the agent.
- **Reward** should align with the real goal, not a shortcut.
- **Simulation** is how we test whether our formulation behaves sensibly.

### Common mistake to avoid

A reward can be mathematically clear but educationally wrong if it teaches the wrong behavior.

### Next step

Move to `07_mini_projects_cartpole_frozenlake_qlearning_dqn.ipynb` to connect these design ideas to simple applied examples.

## 📚 References & Further Reading

**Books:**
- Sutton & Barto — [Reinforcement Learning: An Introduction](http://incompleteideas.net/book/the-book-2nd.html) (free online, the RL bible)

**Papers:**
- Mnih et al. (2015) — [Human-level control through deep RL (DQN)](https://www.nature.com/articles/nature14236)
- Silver et al. (2016) — [AlphaGo](https://www.nature.com/articles/nature16961)

**State-of-the-Art:** OpenAI Five beat world champions in Dota2; AlphaFold uses RL-like optimization for protein folding.

## Closing Takeaway

**Teaching takeaway:** Good RL work begins with problem formulation: choosing a useful state, a realistic action space, and a reward that matches the real goal.

**If students remember one idea:** Many RL failures come from poor problem definition, not from the algorithm itself.

**Quick check before moving on:**
- Can you redesign a bad reward so it better reflects the true objective?
- Can you explain why missing state information can make learning impossible?

**Bridge to the next step:** The mini-project lesson shows how these design choices appear in practical RL experiments.
